# Four-Agent Stair Arena API Probe

This notebook checks the live Stair AI Arena connections for all four agents: `monk`, `anchor`, `hunter`, and `blitz`.

It is default-safe. It reads identity, wallet, matches, orders, exposure, settlement shape, and ledger validation. It builds order payloads but does not place a real order unless you explicitly set `ALLOW_REAL_ORDER_POST = True` and provide a fixture/team/price.

## Safety Defaults

- `ALLOW_REAL_ORDER_POST = False` by default.
- Ledger `validate` is used by default, not ledger `batch` submit.
- Order payloads are printed before any optional POST.
- Use a tiny `ORDER_USD_SIZE`, usually `$1.00`, if you deliberately test a real order.

In [7]:
from __future__ import annotations

import json
import os
import time
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import httpx
from dotenv import load_dotenv

# Load repo .env when this notebook is opened from jupyter/.
load_dotenv(Path('..') / '.env', override=True)

ARENA_BASE = os.getenv('ARENA_BASE', 'https://stair-ai.com').rstrip('/')
ARENA_API = f'{ARENA_BASE}/api/v1/arena'

AGENT_KEYS = {
    'monk': os.getenv('AGENT_KEY_MONK', '').strip(),
    'anchor': os.getenv('AGENT_KEY_ANCHOR', '').strip(),
    'hunter': os.getenv('AGENT_KEY_HUNTER', '').strip(),
    'blitz': os.getenv('AGENT_KEY_BLITZ', '').strip(),
}

# Optional live order probe. Keep False unless you intentionally want a real order POST.
ALLOW_REAL_ORDER_POST = True
ORDER_FIXTURE_ID = os.getenv('ORDER_FIXTURE_ID', '19609127')
ORDER_TEAM_CODE = os.getenv('ORDER_TEAM_CODE', 'draw')
ORDER_USD_SIZE = '1.00'
ORDER_LIMIT_PRICE = 0.01
ORDER_TIF_SECONDS = 30

TIMEOUT = 20

def headers(api_key: str) -> dict[str, str]:
    return {'x-api-key': api_key, 'Content-Type': 'application/json'}

def redacted(key: str) -> str:
    return f'{key[:8]}...{key[-4:]}' if key else '(missing)'

def pretty(obj: Any) -> None:
    print(json.dumps(obj, indent=2, default=str)[:5000])

def request(method: str, path: str, api_key: str, *, params=None, body=None, timeout=TIMEOUT) -> dict:
    url = f'{ARENA_API}{path}'
    try:
        r = httpx.request(method, url, headers=headers(api_key), params=params, json=body, timeout=timeout)
        try:
            payload = r.json()
        except Exception:
            payload = r.text[:1000]
        return {'ok': r.is_success, 'status_code': r.status_code, 'url': url, 'body': payload}
    except Exception as exc:
        return {'ok': False, 'status_code': 0, 'url': url, 'error': repr(exc)}

print('Arena:', ARENA_BASE)
for name, key in AGENT_KEYS.items():
    print(f'{name:8s}', redacted(key))

Arena: https://stair-ai.com
monk     bAIFKoaz...BxXM
anchor   TJSHcJPm...hYoI
hunter   IyfSgNWd...RRJJ
blitz    bHBUlrwm...VuIH


## 0b. Read-Only Data Connection Smoke Test

Checks Supabase plus the live data/API connections the agents depend on. This cell is read-only: no orders, no ledger batch submit, no state writes. It prints a compact status table so you can see which upstreams are actually reachable from the notebook runtime.


In [ ]:
# Read-only connection smoke tests for agent data inputs.
# Safe to run: only GET/read-style calls, no orders, no ledger batch submit.
import datetime as _dt

ROOT = Path('..').resolve()
if str(ROOT) not in os.sys.path:
    os.sys.path.insert(0, str(ROOT))

import config
from data import reddit_sentiment, supabase_client, web_search

DATA_PROXY_API = f"{ARENA_BASE}/api/v1/data/proxy"
WEB_API = f"{ARENA_BASE}/api/v1/web"

CONNECTION_FIXTURE_ID = str(os.getenv('CONNECTION_FIXTURE_ID', ORDER_FIXTURE_ID))
CONNECTION_HOME = os.getenv('CONNECTION_HOME', 'Spain')
CONNECTION_AWAY = os.getenv('CONNECTION_AWAY', 'Cape Verde Islands')
CONNECTION_DATE = os.getenv('CONNECTION_DATE', _dt.date.today().isoformat())

connection_results = {}

def record_connection(name: str, ok: bool, detail: str = '', **extra):
    connection_results[name] = {'ok': bool(ok), 'detail': detail, **extra}
    mark = 'OK ' if ok else 'BAD'
    print(f'{mark} {name:18s} {detail}')

def safe_call(name, fn):
    try:
        return fn()
    except Exception as exc:
        record_connection(name, False, f'{type(exc).__name__}: {exc}')
        return None

print('Fixture probe:', CONNECTION_FIXTURE_ID)
print('Team probe:', CONNECTION_HOME, 'vs', CONNECTION_AWAY)
print('Reader key:', redacted(reader_key) if reader_key else '(missing)')
print()

# 1) Supabase public catalog and arena prior schema.
def _check_supabase():
    catalog = supabase_client.get_catalog()
    record_connection('supabase catalog', bool(catalog), f'{len(catalog)} catalog rows')
    cid_home = supabase_client.resolve_country_id(CONNECTION_HOME)
    cid_away = supabase_client.resolve_country_id(CONNECTION_AWAY)
    known_gap = supabase_client.known_missing_country(CONNECTION_HOME) or supabase_client.known_missing_country(CONNECTION_AWAY)
    record_connection(
        'supabase priors', bool(cid_home or cid_away or known_gap),
        f'home_id={cid_home} away_id={cid_away} known_gap={known_gap}',
    )
_check_supabase()

# 2) Stair Arena agent/read endpoints.
if reader_key:
    me = request('GET', '/agents/me', reader_key)
    record_connection('arena agents/me', me['ok'], f"http={me['status_code']}")
    matches_probe = request('GET', '/matches', reader_key)
    body = matches_probe.get('body')
    n_matches = len(body.get('matches', [])) if isinstance(body, dict) else len(body or []) if isinstance(body, list) else 0
    record_connection('arena matches', matches_probe['ok'], f"http={matches_probe['status_code']} matches={n_matches}")
else:
    record_connection('arena agents/me', False, 'no reader_key')
    record_connection('arena matches', False, 'no reader_key')

# 3) Sportmonks proxy through Stair data proxy.
if reader_key:
    sportmonks = httpx.get(
        f'{DATA_PROXY_API}/sportmonks/v3/football/fixtures/{CONNECTION_FIXTURE_ID}',
        headers={'x-api-key': reader_key},
        params={'include': 'participants;predictions;odds;xGFixture'},
        timeout=60,
    )
    record_connection('sportmonks proxy', sportmonks.is_success, f'http={sportmonks.status_code}')
else:
    record_connection('sportmonks proxy', False, 'no reader_key')

# 4) Polymarket mapping/proxy through Stair.
if reader_key:
    pm_map = httpx.get(
        f'{WEB_API}/mapping',
        headers={'x-api-key': reader_key},
        params={'fixture_id': CONNECTION_FIXTURE_ID},
        timeout=20,
    )
    record_connection('polymarket map', pm_map.is_success, f'http={pm_map.status_code}')
else:
    record_connection('polymarket map', False, 'no reader_key')

# 5) BZZOIRO direct API config/reachability.
def _check_bzzoiro():
    if not config.BZZOIRO_ENABLED:
        record_connection('bzzoiro', False, 'BZZOIRO_ENABLED false or no key')
        return
    url = f'{config.BZZOIRO_API}/v2/events/'
    res = httpx.get(url, headers={'Authorization': f'Token {config.BZZOIRO_KEY}'}, params={'limit': 1}, timeout=config.BZZOIRO_TIMEOUT_SECONDS)
    record_connection('bzzoiro', res.is_success, f'http={res.status_code}')
_check_bzzoiro()

# 6) Web search backend and Reddit fallback path.
def _check_web_reddit():
    web = web_search.gather_research(CONNECTION_HOME, CONNECTION_AWAY, CONNECTION_DATE)
    record_connection('web search', bool(web.get('total_results')), f"backend={web.get('backend')} results={web.get('total_results')}")
    reddit = reddit_sentiment.get_sentiment_bundle(CONNECTION_HOME, CONNECTION_AWAY)
    reddit_ok = bool(reddit.get('threads_found') or reddit.get('comments_found') or reddit.get('top_comments'))
    record_connection(
        'reddit sentiment', reddit_ok,
        f"source={reddit.get('source')} threads={reddit.get('threads_found')} comments={reddit.get('comments_found')}",
    )
_check_web_reddit()

# 7) Grok/XAI config check. This is intentionally not a model call.
record_connection('grok/xai key', bool(config.XAI_KEY), 'configured' if config.XAI_KEY else 'missing XAI_KEY/XAI_API_KEY/GROK_API_KEY')

# 8) odds2prob service health-ish check. It may be unavailable if the hosted converter is down.
def _check_odds2prob():
    url = f"{config.ODDS2PROB_URL.rstrip('/')}/health"
    res = httpx.get(url, timeout=10)
    record_connection('odds2prob', res.status_code < 500, f'http={res.status_code}')
_check_odds2prob()

print('\nConnection result JSON:')
pretty(connection_results)


## 1. Check All Four Agent Identities And Wallets

In [2]:
identity_results = {}

for agent, key in AGENT_KEYS.items():
    if not key:
        identity_results[agent] = {'ok': False, 'error': f'AGENT_KEY_{agent.upper()} missing'}
        continue
    res = request('GET', '/agents/me', key)
    identity_results[agent] = res
    body = res.get('body') if isinstance(res.get('body'), dict) else {}
    wallet = body.get('wallet') or {}
    print(
        f"{agent:8s} ok={res['ok']} http={res['status_code']} "
        f"name={body.get('display_name')} phase={body.get('lifecycle_phase')} "
        f"available={wallet.get('available_balance_usdc')} locked={wallet.get('locked_balance_usdc')}"
    )

identity_results

monk     ok=True http=200 name=GiggleGambler phase=active available=100.000000 locked=0
anchor   ok=True http=200 name=Qdoba phase=active available=106.993982 locked=0
hunter   ok=True http=200 name=CHI CHI CHILE phase=active available=117.509519 locked=0
blitz    ok=True http=200 name=WC phase=active available=129.607672 locked=0


{'monk': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/agents/me',
  'body': {'agent_id': '769eb4f0-ede0-466c-bb87-19b6472b77b1',
   'avatar_url': None,
   'bio': None,
   'created_at': 1780717766567,
   'creator_handle': None,
   'display_name': 'GiggleGambler',
   'lifecycle_phase': 'active',
   'slug': 'gigglegambler',
   'wallet': {'address': '0xB3ffceAa29215a4299FeC51F7849203a993df585',
    'available_balance_usdc': '100.000000',
    'funder_address': '0x3f885B74D0891f20809FDBb2650Cba79795D8683',
    'locked_balance_usdc': '0',
    'polymarket_profile_url': 'https://polymarket.com/profile/0x3f885B74D0891f20809FDBb2650Cba79795D8683',
    'polyscan_url': 'https://polygonscan.com/address/0x3f885B74D0891f20809FDBb2650Cba79795D8683',
    'wallet_id': '33b7bb47-c920-4238-bb4f-865b173e9abb'}}},
 'anchor': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/agents/me',
  'body': {'agent_id': '04a98611-64a0-472b-a530-42c29b1def4b'

## 2. Shared Match Timing Endpoints

Uses the first configured agent key. This is what the live runner uses to confirm windows before betting.

In [3]:
reader_agent = next((name for name, key in AGENT_KEYS.items() if key), None)
reader_key = AGENT_KEYS[reader_agent] if reader_agent else ''
assert reader_key, 'No agent keys configured.'

matches_res = request('GET', '/matches', reader_key)
print('GET /matches', matches_res['ok'], matches_res['status_code'])
matches = (matches_res.get('body') or {}).get('matches') if isinstance(matches_res.get('body'), dict) else None
if matches is None and isinstance(matches_res.get('body'), list):
    matches = matches_res['body']
matches = matches or []
print('match count:', len(matches))
pretty(matches[:3])

probe_fixture_id = str((matches[0] or {}).get('fixture_id') or (matches[0] or {}).get('id') or ORDER_FIXTURE_ID) if matches else ORDER_FIXTURE_ID
match_res = request('GET', f'/matches/{probe_fixture_id}', reader_key)
print('GET /matches/{fixture_id}', match_res['ok'], match_res['status_code'], 'fixture', probe_fixture_id)
pretty(match_res.get('body'))

GET /matches True 200
match count: 104
[
  {
    "current_window": null,
    "fixture_id": "19609127",
    "ht_enabled": false,
    "ht_lock_utc": 1781208000000,
    "ht_open_utc": 1781207100000,
    "kickoff_utc": 1781204400000,
    "pre_match_lock_utc": 1781204400000,
    "prematch_enabled": true,
    "server_ts_utc": 1781545486081,
    "status": "scheduled"
  },
  {
    "current_window": null,
    "fixture_id": "19609153",
    "ht_enabled": false,
    "ht_lock_utc": 1781233200000,
    "ht_open_utc": 1781232300000,
    "kickoff_utc": 1781229600000,
    "pre_match_lock_utc": 1781229600000,
    "prematch_enabled": true,
    "server_ts_utc": 1781545486081,
    "status": "scheduled"
  },
  {
    "current_window": null,
    "fixture_id": "19609154",
    "ht_enabled": false,
    "ht_lock_utc": 1781294400000,
    "ht_open_utc": 1781293500000,
    "kickoff_utc": 1781290800000,
    "pre_match_lock_utc": 1781290800000,
    "prematch_enabled": true,
    "server_ts_utc": 1781545486081,
    "stat

## 3. Orders And Exposure Read Checks For Each Agent

This does not create orders. It confirms the split API contract: `/orders` lists orders, `/exposure` lists holdings.

In [4]:
readback = {}

for agent, key in AGENT_KEYS.items():
    if not key:
        continue
    orders = request('GET', '/orders', key)
    exposure = request('GET', '/exposure', key)
    readback[agent] = {'orders': orders, 'exposure': exposure}
    orders_body = orders.get('body')
    exposure_body = exposure.get('body')
    order_count = len(orders_body.get('orders') or orders_body.get('data') or []) if isinstance(orders_body, dict) else (len(orders_body) if isinstance(orders_body, list) else 'n/a')
    exposure_count = len(exposure_body.get('positions') or exposure_body.get('data') or []) if isinstance(exposure_body, dict) else (len(exposure_body) if isinstance(exposure_body, list) else 'n/a')
    print(f'{agent:8s} orders http={orders["status_code"]} count={order_count} | exposure http={exposure["status_code"]} count={exposure_count}')

readback

monk     orders http=200 count=0 | exposure http=200 count=0
anchor   orders http=200 count=2 | exposure http=200 count=1
hunter   orders http=200 count=3 | exposure http=200 count=2
blitz    orders http=200 count=10 | exposure http=200 count=6


{'monk': {'orders': {'ok': True,
   'status_code': 200,
   'url': 'https://stair-ai.com/api/v1/arena/orders',
   'body': {'orders': []}},
  'exposure': {'ok': True,
   'status_code': 200,
   'url': 'https://stair-ai.com/api/v1/arena/exposure',
   'body': {'positions': [], 'source': 'data-api', 'unmapped': []}}},
 'anchor': {'orders': {'ok': True,
   'status_code': 200,
   'url': 'https://stair-ai.com/api/v1/arena/orders',
   'body': {'orders': [{'fixture_id': '19609158',
      'open_avg_fill_price': 0.039,
      'order_id': 'd20205f6-8703-4db0-b039-462fa7658425',
      'status': 'settled',
      'team_code': 'draw',
      'usd_size_filled': '2.009999',
      'window': 'PRE_MATCH'},
     {'fixture_id': '19609156',
      'open_avg_fill_price': 0.18,
      'order_id': '50194c6c-f95f-4108-b0d9-9fdb6c72d3da',
      'status': 'settled',
      'team_code': 'AUS',
      'usd_size_filled': '1.999999',
      'window': 'PRE_MATCH'}]}},
  'exposure': {'ok': True,
   'status_code': 200,
   'url': '

## 4. Build Real Order Payloads For All Agents Without Posting

This verifies the wire shape from the 2026-06-10 notebook/docs: `fixture_id`, `team_code`, string `usd_size`, `limit_price`, `time_in_force_seconds`, and `idempotency_key`.

In [5]:
order_payloads = {}

for agent, key in AGENT_KEYS.items():
    if not key:
        continue
    payload = {
        'fixture_id': str(ORDER_FIXTURE_ID),
        'team_code': ORDER_TEAM_CODE,
        'usd_size': f'{float(ORDER_USD_SIZE):.2f}',
        'limit_price': round(float(ORDER_LIMIT_PRICE), 4),
        'time_in_force_seconds': int(ORDER_TIF_SECONDS),
        'idempotency_key': str(uuid.uuid4()),
    }
    order_payloads[agent] = payload

pretty(order_payloads)

{
  "monk": {
    "fixture_id": "19609127",
    "team_code": "draw",
    "usd_size": "1.00",
    "limit_price": 0.01,
    "time_in_force_seconds": 30,
    "idempotency_key": "489f6b1e-d1fa-4918-a89b-8c8dd256e9d9"
  },
  "anchor": {
    "fixture_id": "19609127",
    "team_code": "draw",
    "usd_size": "1.00",
    "limit_price": 0.01,
    "time_in_force_seconds": 30,
    "idempotency_key": "99a7a78c-8635-492c-a719-af3c89eb206b"
  },
  "hunter": {
    "fixture_id": "19609127",
    "team_code": "draw",
    "usd_size": "1.00",
    "limit_price": 0.01,
    "time_in_force_seconds": 30,
    "idempotency_key": "d9a3709d-336c-4ed5-ae8c-c5b12feaad55"
  },
  "blitz": {
    "fixture_id": "19609127",
    "team_code": "draw",
    "usd_size": "1.00",
    "limit_price": 0.01,
    "time_in_force_seconds": 30,
    "idempotency_key": "312fa197-d998-42c4-b4d8-25a981287518"
  }
}


## 5. Optional Real Order POST Probe

This is deliberately blocked unless `ALLOW_REAL_ORDER_POST = True`. If enabled, it posts one tiny order per configured agent using the payloads above, then polls each returned order id. Do not enable this unless a prediction window is open and you intentionally want real play-money orders.

In [8]:
order_post_results = {}

if not ALLOW_REAL_ORDER_POST:
    print('Blocked: set ALLOW_REAL_ORDER_POST = True to place real Stair order probes.')
else:
    for agent, payload in order_payloads.items():
        key = AGENT_KEYS[agent]
        print(f'POST /orders for {agent}:')
        pretty(payload)
        post = request('POST', '/orders', key, body=payload, timeout=60)
        result = {'post': post, 'polls': []}
        order_id = post.get('body', {}).get('order_id') if isinstance(post.get('body'), dict) else None
        if order_id:
            for _ in range(6):
                time.sleep(5)
                poll = request('GET', f'/orders/{order_id}', key)
                result['polls'].append(poll)
                status = poll.get('body', {}).get('status') if isinstance(poll.get('body'), dict) else None
                print(agent, order_id, status)
                if status in {'filled', 'closed', 'rejected', 'cancelled', 'expired'}:
                    break
        order_post_results[agent] = result

order_post_results

POST /orders for monk:
{
  "fixture_id": "19609127",
  "team_code": "draw",
  "usd_size": "1.00",
  "limit_price": 0.01,
  "time_in_force_seconds": 30,
  "idempotency_key": "489f6b1e-d1fa-4918-a89b-8c8dd256e9d9"
}
POST /orders for anchor:
{
  "fixture_id": "19609127",
  "team_code": "draw",
  "usd_size": "1.00",
  "limit_price": 0.01,
  "time_in_force_seconds": 30,
  "idempotency_key": "99a7a78c-8635-492c-a719-af3c89eb206b"
}
POST /orders for hunter:
{
  "fixture_id": "19609127",
  "team_code": "draw",
  "usd_size": "1.00",
  "limit_price": 0.01,
  "time_in_force_seconds": 30,
  "idempotency_key": "d9a3709d-336c-4ed5-ae8c-c5b12feaad55"
}
POST /orders for blitz:
{
  "fixture_id": "19609127",
  "team_code": "draw",
  "usd_size": "1.00",
  "limit_price": 0.01,
  "time_in_force_seconds": 30,
  "idempotency_key": "312fa197-d998-42c4-b4d8-25a981287518"
}


{'monk': {'post': {'ok': False,
   'status_code': 400,
   'url': 'https://stair-ai.com/api/v1/arena/orders',
   'body': {'defined': False,
    'code': 'BAD_REQUEST',
    'status': 400,
    'message': 'Fixture is not in an open trading window',
    'data': {'code': 'fixture_not_open'}}},
  'polls': []},
 'anchor': {'post': {'ok': False,
   'status_code': 400,
   'url': 'https://stair-ai.com/api/v1/arena/orders',
   'body': {'defined': False,
    'code': 'BAD_REQUEST',
    'status': 400,
    'message': 'Fixture is not in an open trading window',
    'data': {'code': 'fixture_not_open'}}},
  'polls': []},
 'hunter': {'post': {'ok': False,
   'status_code': 400,
   'url': 'https://stair-ai.com/api/v1/arena/orders',
   'body': {'defined': False,
    'code': 'BAD_REQUEST',
    'status': 400,
    'message': 'Fixture is not in an open trading window',
    'data': {'code': 'fixture_not_open'}}},
  'polls': []},
 'blitz': {'post': {'ok': False,
   'status_code': 400,
   'url': 'https://stair-ai.

## 6. Ledger Validation Probe For Each Agent

This uses `/ledger/records/validate`, so it checks schema and prediction shape without persisting records.

In [9]:
def ledger_record(session_id: str, behavior: str, **fields) -> dict:
    rec = {
        'schema_version': '0.3',
        'session_id': session_id,
        'record_id': str(uuid.uuid4()),
        'behavior': behavior,
        'client_ts_utc': int(time.time() * 1000),
    }
    rec.update({k: v for k, v in fields.items() if v is not None})
    return rec

ledger_validation = {}

for agent, key in AGENT_KEYS.items():
    if not key:
        continue
    session_id = f'api-probe:{ORDER_FIXTURE_ID}:{agent}:{int(time.time())}'
    records = [
        ledger_record(
            session_id, 'Observing',
            trigger_source='notebook_probe',
            trigger_type='cron_trigger',
            trigger_description=f'API probe for {agent}',
            trigger_payload_summary=f'fixture_id={ORDER_FIXTURE_ID}; no order placed by this validation cell',
        ),
        ledger_record(
            session_id, 'Acting',
            action_type='prediction',
            target_system='arena',
            action_summary=f'Validate prediction record for {ORDER_FIXTURE_ID}',
            parameters={'fixture_id': str(ORDER_FIXTURE_ID), 'outcome': ORDER_TEAM_CODE, 'probability': 0.333},
            dry_run=False,
            execution_status='confirmed',
        ),
    ]
    res = request('POST', '/ledger/records/validate', key, body={'records': records}, timeout=30)
    ledger_validation[agent] = res
    print(f'{agent:8s} validate ok={res["ok"]} http={res["status_code"]}')
    pretty(res.get('body'))

ledger_validation

monk     validate ok=True http=200
{
  "errors": [
    {
      "code": "prediction_window_closed",
      "index": 1,
      "message": "No prediction window is currently open for this fixture"
    }
  ],
  "valid": false
}
anchor   validate ok=True http=200
{
  "errors": [
    {
      "code": "prediction_window_closed",
      "index": 1,
      "message": "No prediction window is currently open for this fixture"
    }
  ],
  "valid": false
}
hunter   validate ok=True http=200
{
  "errors": [
    {
      "code": "prediction_window_closed",
      "index": 1,
      "message": "No prediction window is currently open for this fixture"
    }
  ],
  "valid": false
}
blitz    validate ok=True http=200
{
  "errors": [
    {
      "code": "prediction_window_closed",
      "index": 1,
      "message": "No prediction window is currently open for this fixture"
    }
  ],
  "valid": false
}


{'monk': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/ledger/records/validate',
  'body': {'errors': [{'code': 'prediction_window_closed',
     'index': 1,
     'message': 'No prediction window is currently open for this fixture'}],
   'valid': False}},
 'anchor': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/ledger/records/validate',
  'body': {'errors': [{'code': 'prediction_window_closed',
     'index': 1,
     'message': 'No prediction window is currently open for this fixture'}],
   'valid': False}},
 'hunter': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/ledger/records/validate',
  'body': {'errors': [{'code': 'prediction_window_closed',
     'index': 1,
     'message': 'No prediction window is currently open for this fixture'}],
   'valid': False}},
 'blitz': {'ok': True,
  'status_code': 200,
  'url': 'https://stair-ai.com/api/v1/arena/ledger/records/validate',
  'body': {'error

## 7. Optional Ledger Session Bind Probe

This can persist a session fixture binding but does not submit ledger records. Leave disabled unless you want to verify the bind endpoint directly.

In [ ]:
ALLOW_LEDGER_BIND_POST = False
ledger_bind_results = {}

if not ALLOW_LEDGER_BIND_POST:
    print('Blocked: set ALLOW_LEDGER_BIND_POST = True to POST session fixture bindings.')
else:
    for agent, key in AGENT_KEYS.items():
        if not key:
            continue
        session_id = f'api-probe-bind:{ORDER_FIXTURE_ID}:{agent}:{int(time.time())}'
        res = request('POST', f'/ledger/sessions/{session_id}/fixture', key, body={'fixture_id': str(ORDER_FIXTURE_ID)}, timeout=30)
        ledger_bind_results[agent] = res
        print(f'{agent:8s} bind ok={res["ok"]} http={res["status_code"]}')
        pretty(res.get('body'))

ledger_bind_results

## 8. Settlement Endpoint Shape Probe

This is a read-only check. It may return unresolved/empty data for future fixtures.

In [ ]:
settlement_res = request('GET', f'/polymarket/markets/{ORDER_FIXTURE_ID}/settlement', reader_key)
print('settlement ok=', settlement_res['ok'], 'http=', settlement_res['status_code'])
pretty(settlement_res.get('body'))